# 🎯 04B — SetFit Fine-tuning IndoBERT (Opsi B)
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

### Mengapa SetFit?

Fine-tuning BERT standar membutuhkan ribuan labeled pair. Data kita hanya 277 skripsi.
**SetFit (Sentence Transformer Fine-Tuning)** dirancang khusus untuk skenario ini:

| Aspek | Fine-tuning BERT biasa | SetFit |
|-------|----------------------|--------|
| Data yang dibutuhkan | Ribuan contoh | Puluhan–ratusan contoh ✅ |
| Waktu training | Berjam-jam | 10–30 menit ✅ |
| VRAM | ~8GB+ | ~4GB ✅ |
| Kompleksitas | Tinggi | Sedang ✅ |

### Cara Kerja SetFit

```
Tahap 1 — Contrastive Fine-tuning:
  Pasangan (judul_skripsi, profil_dosen_relevan)  → label POSITIF
  Pasangan (judul_skripsi, profil_dosen_lain)     → label NEGATIF
  → Model belajar mendekatkan pasangan relevan & menjauhkan yang tidak

Tahap 2 — Classification Head (opsional, kita skip):
  → Kita tetap pakai Cosine Similarity (bukan classifier)
     karena tugasnya ranking/rekomendasi, bukan klasifikasi biner
```

> ⚠️ **Aktifkan GPU:** Runtime → Change runtime type → T4 GPU
> ⚠️ **Jalankan 02_preprocessing.ipynb (Opsi B patch) sebelum notebook ini**

---
## 🔧 LANGKAH 0 — Setup & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))
print('✅ Drive ter-mount.')

In [ ]:
!pip install -q setfit sentence-transformers datasets
print('✅ Library siap.')

In [ ]:
import config
import pandas as pd
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
from itertools import combinations
from datasets import Dataset
from setfit import SetFitModel, SetFitTrainer, sample_dataset
from sentence_transformers import SentenceTransformer, losses
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')
if device == 'cuda':
    print(f'   GPU  : {torch.cuda.get_device_name(0)}')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

---
## 📌 LANGKAH 1 — Load Data (termasuk Labeled Pairs dari patch 02B)

In [ ]:
df_skripsi = pd.read_csv(config.FILE_SKRIPSI_CLEAN)
df_dosen   = pd.read_csv(config.FILE_DOSEN_CLEAN)

# Load labeled pairs yang dibuat di patch 02B
pairs_path = os.path.join(config.DATA_PROCESSED, 'labeled_pairs.csv')
df_pairs   = pd.read_csv(pairs_path)

NAMA_DOSEN   = df_dosen['nama_dosen'].tolist()
PROFIL_DOSEN = df_dosen['profil_bersih_bert'].fillna('').tolist()

print('✅ Data dimuat.')
print(f'   Skripsi       : {len(df_skripsi)}')
print(f'   Dosen         : {len(df_dosen)}')
print(f'   Labeled pairs : {len(df_pairs)}')
print()
print('Distribusi label:')
print(df_pairs['label'].value_counts().to_string())
print()
df_pairs.head()

---
## 📌 LANGKAH 2 — Buat Dataset untuk SetFit

In [ ]:
# ─── FORMAT DATASET SETFIT ────────────────────────────────────────
# SetFit mengharapkan format:
#   - 'text'  : teks yang akan di-encode
#   - 'label' : integer (0 atau 1)
#
# Kita encode GABUNGAN judul_skripsi + profil_dosen sebagai satu teks
# lalu SetFit belajar: apakah pasangan ini relevan (1) atau tidak (0)?

# Gabungkan teks pasangan dengan separator [SEP]
df_pairs['text'] = df_pairs['teks_skripsi'] + ' [SEP] ' + df_pairs['teks_dosen']

# Split train / val  (80:20 dari labeled pairs)
df_train_pairs, df_val_pairs = train_test_split(
    df_pairs, test_size=0.2, random_state=42, stratify=df_pairs['label']
)

# Konversi ke HuggingFace Dataset
train_dataset = Dataset.from_pandas(df_train_pairs[['text','label']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(df_val_pairs[['text','label']].reset_index(drop=True))

print(f'✅ Dataset SetFit siap.')
print(f'   Train : {len(train_dataset)} pasangan')
print(f'   Val   : {len(val_dataset)} pasangan')
print(f'   Rasio positif/negatif (train): {df_train_pairs["label"].value_counts().to_dict()}')

---
## 📌 LANGKAH 3 — Load Model SetFit & Training

In [ ]:
# ─── BASE MODEL ───────────────────────────────────────────────────
# Gunakan Indo Sentence-BERT sebagai base model untuk fine-tuning
# Alasan: sudah dilatih pada kalimat Indonesia → lebih efisien
BASE_MODEL = 'firqaaa/indo-sentence-bert-base'

print(f'⏳ Memuat base model: {BASE_MODEL}...')
model_setfit = SetFitModel.from_pretrained(BASE_MODEL)
print(f'✅ Model dimuat.')

In [ ]:
# ─── KONFIGURASI TRAINER ─────────────────────────────────────────
trainer = SetFitTrainer(
    model           = model_setfit,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    loss_class      = losses.CosineSimilarityLoss,  # cocok untuk similarity task
    metric          = 'accuracy',
    batch_size      = 16,
    num_iterations  = 20,    # jumlah pasangan contrastive per sample — naikkan jika data sedikit
    num_epochs      = 3,     # epoch fine-tuning
    column_mapping  = {'text': 'text', 'label': 'label'},
)

print('✅ Trainer siap.')
print(f'   Base model     : {BASE_MODEL}')
print(f'   Loss function  : CosineSimilarityLoss')
print(f'   Batch size     : 16')
print(f'   Num iterations : 20')
print(f'   Num epochs     : 3')
print()
print('⏳ Estimasi waktu training: ~10-20 menit dengan T4 GPU.')
print('   Jangan tutup tab Colab selama proses berlangsung!')

In [ ]:
# ─── JALANKAN TRAINING ────────────────────────────────────────────
print('🚀 Memulai fine-tuning SetFit...\n')
trainer.train()
print('\n✅ Fine-tuning selesai!')

In [ ]:
# ─── EVALUASI PADA VALIDATION SET ────────────────────────────────
metrics = trainer.evaluate()
print(f'📊 Validation Accuracy (binary relevance): {metrics["accuracy"]*100:.2f}%')
print('   (Ini accuracy klasifikasi relevan/tidak — beda dengan Top-K rekomendasi)')

In [ ]:
# ─── SIMPAN MODEL ────────────────────────────────────────────────
model_save_path = os.path.join(config.MODELS_DIR, 'setfit_model')
model_setfit.save_pretrained(model_save_path)
print(f'💾 Model SetFit tersimpan: {model_save_path}')

---
## 📌 LANGKAH 4 — Gunakan Model Fine-tuned untuk Embedding

In [ ]:
# ─── AMBIL BODY ENCODER (SENTENCE TRANSFORMER) ───────────────────
# SetFit terdiri dari: [sentence transformer body] + [classification head]
# Untuk rekomendasi kita hanya butuh body encoder-nya
encoder = model_setfit.model_body  # SentenceTransformer yang sudah di-fine-tune

# Encode profil dosen dengan model yang sudah di-fine-tune
print('⏳ Encoding profil dosen dengan model fine-tuned SetFit...')
embeddings_dosen_setfit = encoder.encode(
    PROFIL_DOSEN,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f'\n✅ Shape embedding dosen (SetFit): {embeddings_dosen_setfit.shape}')

# Simpan
setfit_emb_path = os.path.join(config.MODELS_DIR, 'embeddings_dosen_setfit.npy')
np.save(setfit_emb_path, embeddings_dosen_setfit)
print(f'💾 Embedding SetFit dosen tersimpan: {setfit_emb_path}')

---
## 📌 LANGKAH 5 — Evaluasi Top-K Accuracy (SetFit vs Opsi A)

In [ ]:
# ─── SIAPKAN DATA TEST (sama dengan notebook 04A) ────────────────
dosen_valid = set(NAMA_DOSEN)
df_eval     = df_skripsi[df_skripsi['pembimbing'].isin(dosen_valid)].copy().reset_index(drop=True)
_, df_test  = train_test_split(df_eval, test_size=0.2, random_state=42, stratify=df_eval['pembimbing'])
query_texts = df_test['teks_bersih_bert'].tolist()

# Encode query dengan model SetFit
print('⏳ Encoding query test dengan model SetFit...')
q_emb_setfit = encoder.encode(
    query_texts, batch_size=32,
    show_progress_bar=True, normalize_embeddings=True
)
print(f'✅ Shape query embedding: {q_emb_setfit.shape}')

In [ ]:
# ─── HITUNG TOP-K ACCURACY (SetFit) ─────────────────────────────
def evaluasi_embedding_topk(q_embeddings, d_embeddings, df_test, nama_dosen, top_k_list=[1,3,5]):
    sim_matrix = cosine_similarity(q_embeddings, d_embeddings)
    results    = {k: 0 for k in top_k_list}
    for idx, (_, row) in enumerate(df_test.iterrows()):
        gt      = row['pembimbing']
        ranking = np.argsort(sim_matrix[idx])[::-1]
        ordered = [nama_dosen[i] for i in ranking]
        for k in top_k_list:
            if gt in ordered[:k]:
                results[k] += 1
    n = len(df_test)
    return {k: round(v/n*100, 2) for k, v in results.items()}

acc_setfit = evaluasi_embedding_topk(q_emb_setfit, embeddings_dosen_setfit, df_test, NAMA_DOSEN)

print('=' * 58)
print('📊 HASIL EVALUASI — SetFit Fine-tuning (Opsi B)')
print('=' * 58)
print(f'  Top-1 Accuracy : {acc_setfit[1]:.2f}%')
print(f'  Top-3 Accuracy : {acc_setfit[3]:.2f}%')
print(f'  Top-5 Accuracy : {acc_setfit[5]:.2f}%')
print('=' * 58)

In [ ]:
# ─── UPDATE TABEL EVALUASI LENGKAP ───────────────────────────────
df_eval_all = pd.read_csv(config.FILE_EVALUATION)

new_row = pd.DataFrame([{
    'metode'         : 'Indo Sentence-BERT + SetFit Fine-tuning',
    'top1_accuracy'  : acc_setfit[1],
    'top3_accuracy'  : acc_setfit[3],
    'top5_accuracy'  : acc_setfit[5],
    'n_test'         : len(df_test),
}])
df_eval_all = pd.concat([df_eval_all, new_row], ignore_index=True)
df_eval_all.to_csv(config.FILE_EVALUATION, index=False)

print('📊 Tabel evaluasi lengkap (semua metode):')
print(df_eval_all[['metode','top1_accuracy','top3_accuracy','top5_accuracy']].to_string(index=False))

In [ ]:
# ─── VISUALISASI PERBANDINGAN SEMUA METODE ───────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
x     = np.arange(3)
w     = 0.20
cols  = ['top1_accuracy','top3_accuracy','top5_accuracy']
ks    = ['Top-1', 'Top-3', 'Top-5']
palette = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']

for i, (_, row) in enumerate(df_eval_all.iterrows()):
    vals = [row[c] for c in cols]
    bars = ax.bar(x + i*w, vals, width=w, label=row['metode'],
                  color=palette[i % len(palette)], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

mid = (len(df_eval_all) - 1) * w / 2
ax.set_xticks(x + mid)
ax.set_xticklabels(ks, fontsize=12)
ax.set_ylim(0, 118)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Perbandingan Semua Metode: TF-IDF vs BERT Embedding vs SetFit',
             fontweight='bold', fontsize=13)
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)
ax.axhline(100, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'all_methods_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Grafik perbandingan lengkap tersimpan.')

In [ ]:
# ─── DEMO INTERAKTIF ──────────────────────────────────────────────
def rekomendasikan_setfit(judul_query, top_k=5):
    query_emb = encoder.encode([judul_query], normalize_embeddings=True)
    scores    = cosine_similarity(query_emb, embeddings_dosen_setfit).flatten()
    ranking   = np.argsort(scores)[::-1]
    print(f'🎯 [SetFit] Query: "{judul_query}"')
    print(f'📋 Top-{top_k} Rekomendasi:')
    print('-' * 65)
    for rank, i in enumerate(ranking[:top_k], 1):
        bar = '█' * int(scores[i] * 25)
        print(f'  {rank}. {NAMA_DOSEN[i][:40]:<42} {scores[i]:.4f} {bar}')
    print()

JUDUL_QUERY = "Sistem Deteksi Hoaks Berbasis Natural Language Processing Menggunakan IndoBERT"
print('=' * 65)
rekomendasikan_setfit(JUDUL_QUERY, top_k=5)
print('=' * 65)

---
## ✅ Selesai — Ringkasan Notebook 04B (Opsi B)

| Output | Lokasi |
|--------|--------|
| Model SetFit (fine-tuned) | `models/setfit_model/` |
| Embedding SetFit dosen | `models/embeddings_dosen_setfit.npy` |
| Grafik perbandingan semua metode | `results/all_methods_comparison.png` |
| Hasil evaluasi lengkap | `results/evaluation.csv` |

### 🗺️ Langkah Berikutnya:
> **`05_evaluasi.ipynb`** — Analisis mendalam & perbandingan final semua metode